# Hamiltonian Flow Matching - 2D Duffing Double Well

This notebook transports a Gaussian source centered at `(-1, 0)` to a Gaussian target centered at
`(1, 0)` under the 2D Duffing double-well potential

$$V(x,y)=-\frac12 x^2+\frac14 x^4+\frac{\kappa}{2}y^2+\frac14.$$

The constant `+1/4` sets the well minima to zero and does not affect the Hamiltonian force.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../../../'))

import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.distributions import MultivariateNormal
from torchdyn.core import NeuralODE

from torchcfm.hamiltonian import MeanStdBVPGaussianPath, flow_matching_loss
from torchcfm.optimal_transport import OTPlanSampler
from torchcfm.models.models_v2 import MLP
from torchcfm.utils import torch_wrapper

if torch.cuda.is_available() and torch.cuda.device_count() > 2:
    device = torch.device('cuda:2')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')


def to_numpy(x):
    return x.detach().cpu().numpy()

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

dim = 2
kappa = 5.0
delta = 0.1

source_mean = torch.tensor([-1.0, 0.0], device=device)
target_mean = torch.tensor([1.0, 0.0], device=device)

# Training controls. Increase n_dataset/n_epochs/n_iters for production runs.
batch_size = 64
n_dataset = 32
n_epochs = 100
n_iters = 100
n_warmup_iters = 5000
batch_size_warmup = 1024
lr = 5e-4
bvp_sigma = 0.001
n_steps = 40
tol = 1e-2
quadrature_order  = 3
eval_batch = 1000
solve_t_span = torch.linspace(0, 1, n_steps + 1, device=device)

print(f'device: {device}')
print(f'kappa: {kappa}, delta: {delta}')
print(f'source_mean: {source_mean.tolist()}')
print(f'target_mean: {target_mean.tolist()}')
print(f'Gauss-Hermite quadrature nodes: {quadrature_order ** dim}')

## Duffing Double-Well Potential

In [ ]:
class DuffingDoubleWellPotential:
    def __init__(self, kappa=1.0):
        self.kappa = float(kappa)

    def energy(self, q):
        x = q[..., 0]
        y = q[..., 1]
        return -0.5 * x ** 2 + 0.25 * x ** 4 + 0.5 * self.kappa * y ** 2 + 0.25

    def gradient(self, q):
        x = q[..., 0]
        y = q[..., 1]
        return torch.stack([x ** 3 - x, self.kappa * y], dim=-1)

    def linear_gradient(self, q):
        return self.gradient(q)

    def force(self, q):
        return -self.gradient(q)


potential = DuffingDoubleWellPotential(kappa=kappa)

q_test = torch.stack([source_mean, torch.zeros_like(source_mean), target_mean])
print(f'V([-1,0]) = {potential.energy(source_mean.reshape(1, -1)).item():.6f}')
print(f'V([0,0]) = {potential.energy(torch.zeros(1, 2, device=device)).item():.6f}')
print(f'V([1,0]) = {potential.energy(target_mean.reshape(1, -1)).item():.6f}')
print(f'force at [0.5, 0.25] = {potential.force(torch.tensor([[0.5, 0.25]], device=device))}')

## Endpoint Gaussian Distributions

In [ ]:
covariance = delta ** 2 * torch.eye(dim, device=device)
source_dist = MultivariateNormal(source_mean, covariance)
target_dist = MultivariateNormal(target_mean, covariance)


def sample_source(n):
    return source_dist.sample((n,)).to(device)


def sample_target(n):
    return target_dist.sample((n,)).to(device)


def plot_endpoint_samples(x0, x1, title, n_show=1000):
    x0_np = to_numpy(x0[:n_show])
    x1_np = to_numpy(x1[:n_show])
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(x0_np[:, 0], x0_np[:, 1], s=8, alpha=0.45, c='royalblue', label=r'$\rho_0$')
    ax.scatter(x1_np[:, 0], x1_np[:, 1], s=8, alpha=0.45, c='tomato', label=r'$\rho_1$')
    ax.scatter([-1, 1], [0, 0], c=['navy', 'darkred'], s=70, marker='x', label='means')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_title(title)
    ax.axis('equal')
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()


x0_vis = sample_source(1000)
x1_vis = sample_target(1000)
plot_endpoint_samples(x0_vis, x1_vis, 'Gaussian endpoint samples')

## Potential Contours

In [ ]:
def potential_grid(xlim=(-1.8, 1.8), ylim=(-1.2, 1.2), n=220):
    xs = torch.linspace(*xlim, n, device=device)
    ys = torch.linspace(*ylim, n, device=device)
    Y, X = torch.meshgrid(ys, xs, indexing='ij')
    points = torch.stack([X.reshape(-1), Y.reshape(-1)], dim=-1)
    Z = potential.energy(points).reshape(n, n)
    return to_numpy(X), to_numpy(Y), to_numpy(Z)


X, Y, Z = potential_grid()
fig, ax = plt.subplots(figsize=(7, 5))
levels = np.linspace(np.nanmin(Z), np.nanpercentile(Z, 95), 30)
contour = ax.contourf(X, Y, Z, levels=levels, cmap='viridis')
ax.contour(X, Y, Z, levels=levels[::4], colors='white', alpha=0.35, linewidths=0.6)
ax.scatter([-1, 1], [0, 0], c=['royalblue', 'tomato'], s=80, edgecolor='black')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Duffing double-well potential')
ax.axis('equal')
fig.colorbar(contour, ax=ax, label='V(x, y)')
plt.tight_layout()
plt.show()

## BVP Path, Model, and Training Helpers

In [ ]:
ot_sampler = OTPlanSampler(method='exact')

fwd_model = MLP(dim, out_dim=dim, w=256, time_varying=True).to(device)
bwd_model = MLP(dim, out_dim= dim, w = 256, time_varying=True).to(device)
fwd_optimizer = torch.optim.Adam(fwd_model.parameters(), lr=lr)
bwd_optimizer = torch.optim.Adam(bwd_model.parameters(), lr=lr)
fwd_losses = []
bwd_losses = []


def make_node(model):
    return NeuralODE(torch_wrapper(model), sensitivity='adjoint', solver='euler')


def make_path():
    return MeanStdBVPGaussianPath(
        potential,
        sigma=sigma_path,
        n_steps=n_steps,
        tol=tol,
        quadrature_order=quadrature_order,
    )


def solve_paths(x0, x1, label='path'):
    path = make_path()
    print(f'Solving {x0.shape[0]} {label} Duffing mean/std BVPs...')
    states = path.batch_solve(x0, x1)
    keep = path.success_mask.to(device=x0.device)
    x0_keep = x0[keep]
    x1_keep = x1[keep]
    n_failed = int((~keep).sum().item())
    print(f'{label}: kept {x0_keep.shape[0]} / {keep.numel()} BVPs; failed {n_failed}; states: {states.shape}')
    if n_failed:
        preview = list(path.failure_messages.items())[:5]
        print(f'{label}: first failures: {preview}')
    return path, x0_keep, x1_keep, states


def train_on_cached_paths(model, optimizer, path, x0, x1, n_steps_train, label, log_every=500):
    model.train()
    step_losses = []
    n_pairs = x0.shape[0]
    if n_pairs == 0:
        raise RuntimeError(f'{label}: no successful BVP pairs to train on.')

    x0 = x0.to(device)
    x1 = x1.to(device)
    for step in range(n_steps_train):
        optimizer.zero_grad()

        idx = torch.randint(0, n_pairs, (batch_size,), device=device)
        x0_b = x0[idx]
        x1_b = x1[idx]
        t = torch.rand((batch_size, 1), device=device, dtype=x0_b.dtype)
        epsilon = torch.randn_like(x0_b)

        xt = path.sample_xt(x0_b, x1_b, t, epsilon)
        ut = path.compute_ut(x0_b, x1_b, t, xt)
        vt = model(torch.cat([xt, t], dim=-1))
        loss = flow_matching_loss(vt, ut)

        loss.backward()
        optimizer.step()
        step_losses.append(loss.item())

        if step % log_every == 0 or step == n_steps_train - 1:
            print(f'{label} step {step:5d}: loss = {loss.item():.5f}')
    return step_losses

def train_on_ot_pairs(model, optimizer, x0, x1, n_steps_train, label, log_every=500):
    model.train()
    step_losses = []
    n_pairs = x0.shape[0]
    if n_pairs == 0:
        raise RuntimeError(f'{label}: no OT pairs to train on.')

    x0 = x0.to(device)
    x1 = x1.to(device)
    for step in range(n_steps_train):
        optimizer.zero_grad()

        idx = torch.randint(0, n_pairs, (batch_size,), device=device)
        x0_b = x0[idx]
        x1_b = x1[idx]
        t = torch.rand((batch_size, 1), device=device, dtype=x0_b.dtype)

        xt = (1.0 - t) * x0_b + t * x1_b
        ut = x1_b - x0_b
        vt = model(torch.cat([xt, t], dim=-1))
        loss = flow_matching_loss(vt, ut)

        loss.backward()
        optimizer.step()
        step_losses.append(loss.item())

        if step % log_every == 0 or step == n_steps_train - 1:
            print(f'{label} step {step:5d}: loss = {loss.item():.5f}')
    return step_losses

## Initial Cached Gaussian BVP Dataset

In [ ]:
x0_all = sample_source(n_dataset)
x1_all = sample_target(n_dataset)
x0_coupled, x1_coupled = ot_sampler.sample_plan(x0_all, x1_all)
path, x0_coupled, x1_coupled, states = solve_paths(x0_coupled, x1_coupled, label='initial')
n_dataset = x0_coupled.shape[0]

t_state = to_numpy(path.t_grid)
plot_idx = np.linspace(0, n_dataset - 1, min(6, n_dataset)).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].contour(X, Y, Z, levels=np.linspace(np.nanmin(Z), np.nanpercentile(Z, 90), 20), cmap='viridis', alpha=0.7)
for idx in plot_idx:
    mu = to_numpy(states[idx, :, :dim])
    sigma = to_numpy(states[idx, :, 2 * dim])
    axes[0].plot(mu[:, 0], mu[:, 1], linewidth=1.8, alpha=0.8, label=f'path {idx}')
    axes[1].plot(t_state, sigma, linewidth=1.8, alpha=0.8, label=f'path {idx}')

axes[0].scatter([-1, 1], [0, 0], c=['royalblue', 'tomato'], s=60, edgecolor='black')
axes[0].set_title('Cached BVP mean paths')
axes[0].set_xlabel('x')
axes[0].set_ylabel('y')
axes[0].axis('equal')
axes[0].grid(alpha=0.25)
axes[1].set_title('Cached sigma schedules')
axes[1].set_xlabel('t')
axes[1].set_ylabel('sigma(t)')
axes[1].grid(alpha=0.25)
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Training

In [ ]:
warm_x0 = sample_source(batch_size_warmup)
warm_x1 = sample_target(batch_size_warmup)
warm_x0, warm_x1 = ot_sampler.sample_plan(warm_x0, warm_x1)

print('Warm-start forward model on straight OT pairs...')
fwd_losses.extend(train_on_ot_pairs(fwd_model, fwd_optimizer, warm_x0, warm_x1, n_warmup_iters, 'fwd warm'))
print('Warm-start backward model on straight OT pairs...')
bwd_losses.extend(train_on_ot_pairs(bwd_model, bwd_optimizer, warm_x1, warm_x0, n_warmup_iters, 'bwd warm'))

In [ ]:
last_fwd_path = None
last_bwd_path = None
last_fwd_states = None
last_bwd_states = None
last_fwd_x0, last_fwd_x1 = None, None
last_bwd_x0, last_bwd_x1 = None, None

for epoch in range(n_epochs):
    print(f'\n=== Alternating epoch {epoch + 1} / {n_epochs} ===')

    if epoch % 2 == 0:
        # Train fwd using couplings induced by the current bwd NODE.
        y_target = sample_target(n_dataset)
        bwd_model.eval()
        bwd_node = make_node(bwd_model)
        with torch.no_grad():
            bwd_traj = bwd_node.trajectory(y_target, t_span=solve_t_span)
        generated_source = bwd_traj[-1].detach()

        last_fwd_path, last_fwd_x0, last_fwd_x1, last_fwd_states = solve_paths(
            generated_source,
            y_target,
            label=f'epoch {epoch} fwd',
        )
        fwd_losses.extend(
            train_on_cached_paths(
                fwd_model,
                fwd_optimizer,
                last_fwd_path,
                last_fwd_x0,
                last_fwd_x1,
                n_iters,
                label=f'epoch {epoch} fwd',
            )
        )
    else:
        # Train bwd using couplings induced by the current fwd NODE.
        x_source = sample_source(n_dataset)
        fwd_model.eval()
        fwd_node = make_node(fwd_model)
        with torch.no_grad():
            fwd_traj = fwd_node.trajectory(x_source, t_span=solve_t_span)
        generated_target = fwd_traj[-1].detach()

        last_bwd_path, last_bwd_x0, last_bwd_x1, last_bwd_states = solve_paths(
            generated_target,
            x_source,
            label=f'epoch {epoch} bwd',
        )
        bwd_losses.extend(
            train_on_cached_paths(
                bwd_model,
                bwd_optimizer,
                last_bwd_path,
                last_bwd_x0,
                last_bwd_x1,
                n_iters,
                label=f'epoch {epoch} bwd',
            )
        )

In [ ]:
plt.figure(figsize=(8, 3.5))
plt.semilogy(fwd_losses, label='fwd')
plt.semilogy(bwd_losses, label='bwd')
plt.xlabel('model update')
plt.ylabel('flow matching loss')
plt.title('Training losses')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def simulate_model(model, x0, t_span):
    node = make_node(model)
    model.eval()
    with torch.no_grad():
        traj = node.trajectory(x0.to(device), t_span.to(device))
    return traj


eval_t_span = torch.linspace(0, 1, 101, device=device)
x0_eval = sample_source(eval_batch)
x1_eval = sample_target(eval_batch)

fwd_traj = simulate_model(fwd_model, x0_eval, eval_t_span)
bwd_traj = simulate_model(bwd_model, x1_eval, eval_t_span)


def print_gaussian_summary(label, traj, reference_mean):
    terminal = traj[-1]
    generated_mean = terminal.mean(dim=0)
    generated_std = terminal.std(dim=0, unbiased=True).mean().item()
    mean_rmse = (generated_mean - reference_mean).pow(2).mean().sqrt().item()
    print(f'{label} terminal mean RMSE: {mean_rmse:.4f}')
    print(f'{label} average coordinate std: {generated_std:.4f} (target {delta:.4f})')


print_gaussian_summary('Forward', fwd_traj, target_mean)
print_gaussian_summary('Backward', bwd_traj, source_mean)

## Evaluation

In [ ]:
fwd_model.eval()
node = make_node(fwd_model)
eval_t_span = torch.linspace(0, 1, 101, device=device)
x0_eval = sample_source(eval_batch)
x1_ref = sample_target(eval_batch)

with torch.no_grad():
    traj = node.trajectory(x0_eval, t_span=eval_t_span)

terminal = traj[-1]
mean_error = terminal.mean(dim=0) - target_mean
avg_std = terminal.std(dim=0, unbiased=True).mean().item()
print(f'terminal mean RMSE: {mean_error.pow(2).mean().sqrt().item():.4f}')
print(f'terminal average coordinate std: {avg_std:.4f} (target {delta:.4f})')

In [ ]:
def plot_learned_trajectories(traj, x1_ref, n_show=1000, n_lines=160):
    n_show = min(n_show, traj.shape[1], x1_ref.shape[0])
    n_lines = min(n_lines, traj.shape[1])
    traj_np = to_numpy(traj)
    x1_np = to_numpy(x1_ref[:n_show])

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].contourf(X, Y, Z, levels=np.linspace(np.nanmin(Z), np.nanpercentile(Z, 95), 30), cmap='viridis', alpha=0.85)
    axes[0].scatter(traj_np[0, :n_show, 0], traj_np[0, :n_show, 1], s=8, alpha=0.45, c='royalblue', label=r'$\rho_0$')
    axes[0].scatter(x1_np[:, 0], x1_np[:, 1], s=7, alpha=0.18, c='tomato', label=r'reference $\rho_1$')
    axes[0].scatter(traj_np[-1, :n_show, 0], traj_np[-1, :n_show, 1], s=10, alpha=0.55, c='black', marker='x', label='generated terminal')
    axes[0].set_title('Terminal samples vs reference')

    axes[1].contour(X, Y, Z, levels=np.linspace(np.nanmin(Z), np.nanpercentile(Z, 90), 20), cmap='viridis', alpha=0.65)
    for i in range(n_lines):
        axes[1].plot(traj_np[:, i, 0], traj_np[:, i, 1], color='black', alpha=0.14, linewidth=0.8)
    axes[1].scatter(traj_np[0, :n_lines, 0], traj_np[0, :n_lines, 1], s=10, c='royalblue')
    axes[1].scatter(traj_np[-1, :n_lines, 0], traj_np[-1, :n_lines, 1], s=10, c='tomato')
    axes[1].set_title('Learned trajectories')

    for ax in axes:
        ax.scatter([-1, 1], [0, 0], c=['royalblue', 'tomato'], s=60, edgecolor='black')
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.axis('equal')
        ax.grid(alpha=0.25)
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


plot_learned_trajectories(fwd_traj, x1_ref)

## Approximate Hamiltonian Diagnostics

In [ ]:
def trajectory_hamiltonian(traj, t_span):
    dt = t_span[1] - t_span[0]
    velocity = torch.empty_like(traj)
    velocity[0] = (traj[1] - traj[0]) / dt
    velocity[-1] = (traj[-1] - traj[-2]) / dt
    velocity[1:-1] = (traj[2:] - traj[:-2]) / (2.0 * dt)
    kinetic = 0.5 * velocity.pow(2).sum(dim=-1)
    potential_values = torch.stack([potential.energy(traj_i) for traj_i in traj], dim=0)
    return kinetic + potential_values


diagnostic_batch = min(200, fwd_traj.shape[1])
H = trajectory_hamiltonian(fwd_traj[:, :diagnostic_batch], eval_t_span)
H_np = to_numpy(H)

plt.figure(figsize=(8, 4))
plt.plot(to_numpy(eval_t_span), H_np[:, : min(30, diagnostic_batch)], color='0.25', alpha=0.22, linewidth=0.8)
plt.plot(to_numpy(eval_t_span), H_np.mean(axis=1), color='black', linewidth=2.0, label='mean')
plt.xlabel('t')
plt.ylabel('H(t)')
plt.title('Approximate Hamiltonian along learned trajectories')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()